# Feature Engineering

## Categorisation of bets based on the timing of the match
Timing of bets can be placed into 3 major categories:

1. Before the ban/pick phase
2. After the ban/pick phase and before match has started
3. Anytime before the game has ended

For this EDA, we will focus on the first two categories and the last category can be left as a future extension of the project.

In [1]:
%load_ext autoreload
%autoreload 2
from src.postgresql import get_engine
from src.process_data.preprocessing import preprocess_df
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)



In [2]:
engine = get_engine()

In [3]:
df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101616 entries, 0 to 101615
Data columns (total 28 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   0_hero_id        101600 non-null  float64
 1   1_hero_id        101602 non-null  float64
 2   2_hero_id        101606 non-null  float64
 3   3_hero_id        101602 non-null  float64
 4   4_hero_id        101602 non-null  float64
 5   128_hero_id      101600 non-null  float64
 6   129_hero_id      101604 non-null  float64
 7   130_hero_id      101613 non-null  float64
 8   131_hero_id      101604 non-null  float64
 9   132_hero_id      101608 non-null  float64
 10  0_account_id     101600 non-null  float64
 11  1_account_id     101602 non-null  float64
 12  2_account_id     101606 non-null  float64
 13  3_account_id     101602 non-null  float64
 14  4_account_id     101602 non-null  float64
 15  128_account_id   101600 non-null  float64
 16  129_account_id   101604 non-null  floa

# Feature Selection (Manual)

In [4]:
draft_cols = df.filter(like="_hero_id").columns
player_cols = df.filter(like="_account_id").columns
team_cols = ['radiant_name','dire_name']
label_col = 'radiant_win'
time_col = 'start_time'
uuid_col = 'match_id'

In [5]:
df = preprocess_df(df)
df.info()

Removing 2291 rows with missing values
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99325 entries, 0 to 99324
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   0_hero_id       99325 non-null  float64       
 1   1_hero_id       99325 non-null  float64       
 2   2_hero_id       99325 non-null  float64       
 3   3_hero_id       99325 non-null  float64       
 4   4_hero_id       99325 non-null  float64       
 5   128_hero_id     99325 non-null  float64       
 6   129_hero_id     99325 non-null  float64       
 7   130_hero_id     99325 non-null  float64       
 8   131_hero_id     99325 non-null  float64       
 9   132_hero_id     99325 non-null  float64       
 10  0_account_id    99325 non-null  float64       
 11  1_account_id    99325 non-null  float64       
 12  2_account_id    99325 non-null  float64       
 13  3_account_id    99325 non-null  float64       
 14  4_account_id   

# Feature Engineering

## Create Team Level Features

In [2]:
from collections import deque

In [ ]:
def calculate_win_rate(team_histories: list, team_name: str) -> float:
    if not team_histories:
        return 0.5
    
    win = 0
    for match in team_histories:
        if match['radiant_name'] == team_name and match['radiant_win']:
            win += 1
        elif match['dire_name'] == team_name and not match['radiant_win']:
            win += 1
        
    return win/ len(team_histories)
        

In [ ]:


def update_team_history(team_histories, radiant, dire, match):
    if radiant not in team_histories:
        team_histories[radiant] = deque(maxlen=10)
    elif dire not in team_histories:
        team_histories[dire] = deque(maxlen=10)
        
    team_histories[radiant].append(match)
    team_histories[dire].append(match)
    
def update_matchup_history(matchup_histories, radiant, dire, match):
    if (radiant, dire) not in matchup_histories:
        matchup_histories[(radiant, dire)] = deque(maxlen=10)
    
    matchup_histories[(radiant,dire)].append(match)

In [ ]:
def calculate_matchup(matchup_histories: list, team_radiant:str) -> float:
    if not matchup_histories:
        return 0.5
    
    win = 0
    for match in matchup_histories:
        if match['radiant_name'] == team_radiant and match['radiant_win']:
            win += 1
        elif match['dire_name'] == team_radiant and not match['radiant_win']:
            win += 1
    
    return win / len(matchup_histories)

In [ ]:
team_histories = {}
matchup_histories = {}
results = []

for _, match in df.iterrows():
    radiant_team = match['radiant_name']
    dire_team = match['dire_name']
    
    # Calculate features for each row
    radiant_win_rate = calculate_win_rate(team_histories.get(radiant_team, []), radiant_team)
    dire_win_rate = calculate_win_rate(team_histories.get(dire_team, []), dire_team)
    
    all_matches = matchup_histories.get((radiant_team, dire_team), []) + matchup_histories.get((dire_team, radiant_team), [])
    matchup_rate = calculate_matchup(all_matches, radiant_team)
    
    # append features to results
    results.append({
        'match_id': match['match_id'],
        'radiant_win_rate': radiant_win_rate,
        'dire_win_rate': dire_win_rate,
        'radiant_dire_matchup': matchup_rate
    })
    
    # update history dictionaries
    update_team_history(team_histories, radiant_team, dire_team, match)
    update_matchup_history(matchup_histories, radiant_team, dire_team, match)
    
    

key 0
key 1
key 2
key 3
key 4
key 5
key 6
key 7
key 8
key 9


In [ ]:
def last_10_matches_win_rate(df, team_name, current_date):
    last_10_matches = df[((df['radiant_name'] == team_name) | (df['dire_name'] == team_name)) &
                         (df['start_time'] < current_date)].head(10)
                         
    
    if len(last_10_matches) == 0:
        return 0.5 # Impute value as 0.5 for a team's debut match

    wins = ((last_10_matches['radiant_name'] == team_name) & last_10_matches['radiant_win']).sum()
    wins += ((last_10_matches['dire_name'] == team_name) & ~last_10_matches['radiant_win']).sum()

    win_rate = wins / len(last_10_matches)
    return win_rate

In [ ]:

df['radiant_win_rate'] = df.apply(lambda row: last_10_matches_win_rate(df, row['radiant_name'], row['start_time']), axis=1)
df['dire_win_rate'] = df.apply(lambda row: last_10_matches_win_rate(df, row['dire_name'], row['start_time']), axis=1)


In [ ]:
def radiant_dire_matchup(df, radiant_name, dire_name, current_date):
    last_10_matches = df[
        (((df['radiant_name'] == radiant_name) & (df['dire_name'] == dire_name)) |
         ((df['dire_name'] == radiant_name) & (df['radiant_name'] == dire_name))) &
        (df['start_time'] < current_date)].head(10)
    
    if len(last_10_matches) == 0:
        return 0.5 # Impute value as 0.5 for a team's debut match
    
    wins = ((last_10_matches['radiant_name'] == radiant_name) & last_10_matches['radiant_win']).sum()
    wins += ((last_10_matches['dire_name'] == radiant_name) & ~last_10_matches['radiant_win']).sum()

    radiant_dire_matchup = wins / len(last_10_matches)
    
    return radiant_dire_matchup

In [ ]:
df['radiant_dire_matchup'] = df.apply(lambda row: radiant_dire_matchup(df, row['radiant_name'], row['dire_name'], row['start_time']), axis=1)

In [ ]:
team_level_features = df[['radiant_dire_matchup','radiant_win_rate','dire_win_rate', uuid_col]]

In [ ]:
team_level_features

In [ ]:
from sqlmodel import Session
from database.data_models.features import TeamFeatures

In [ ]:
model_fields = {
    name for name, field in TeamFeatures.__fields__.items()
    if not name.startswith('_')
}

model_fields

In [ ]:
# Store to database:

with Session(engine) as session:
    for index, row in team_level_features.iterrows():
        # 'row' is a pandas Series, which works similarly to a dictionary
        filtered_data = {
            field: row[field]
            for field in model_fields
            if field in row
        }
        
        # Create the model instance with the filtered data
        team_features = TeamFeatures(**filtered_data)
        session.merge(team_features)
    
    session.commit()

## Create hero level feature 

In [ ]:
import yaml
CONSTANTS_FILE_PATH = '../constants/constants.yml'
try:
    with open(CONSTANTS_FILE_PATH, 'r') as file:
        data = yaml.safe_load(file) or {}
        hero_dict = data.get('HEROES_CONSTANTS', {})
        if not hero_dict or not isinstance(hero_dict, dict):
            raise ValueError("Unable to load hero constants or they are not in a valid format.")
except FileNotFoundError:
    print(f"'{CONSTANTS_FILE_PATH}' does not exist!")
    
hero_dict

In [ ]:
df[draft_cols] = df[draft_cols].map(hero_dict.get)
df[draft_cols]

In [ ]:
df_melted_heroes = pd.melt(df, id_vars=['start_time','radiant_win','match_id'],
                            value_vars=draft_cols, 
                            var_name='hero_position',
                            value_name='hero_name')

df_melted_heroes

In [ ]:
df_encoded = pd.concat([df_melted_heroes, pd.get_dummies(df_melted_heroes['hero_name'])], axis=1)

df_encoded = df_encoded.groupby(['match_id']).sum(numeric_only=True).reset_index()

df_encoded = df_encoded.drop(columns='radiant_win')

In [ ]:
cols_to_merge = ['start_time', 'radiant_win', 'match_id']
heroes_features = df_encoded.merge(df[cols_to_merge].drop_duplicates(), on='match_id', how='left')
heroes_features

In [ ]:
from sqlmodel import Session
from database.data_models.features import HeroFeatures

In [ ]:
with Session(engine) as session:
    for _, row in heroes_features.iterrows():
        # Filter the row data to only include fields in the model
        # Convert to dict first to make it easier to filter
        row_dict = dict(row)
        match_id = row_dict['match_id']
        hero_picks = []
        
        for column, value in row_dict.items():
            if column not in ['match_id', 'radiant_win', 'start_time']\
            and value == 1 :
                hero_picks.append(column)
                
        hero_features = HeroFeatures(
            match_id=match_id,
            hero_picks=hero_picks
        )
        
        session.merge(hero_features)
    
    session.commit()

## Feature Crossing between players and heros



In [ ]:
df_melted_players = pd.melt(df.copy(), id_vars=['start_time','radiant_win','match_id'],
                            value_vars=player_cols, 
                            var_name='player_position',
                            value_name='account_id')

df_melted_players

In [ ]:
df_melted_players['player_num'] = df_melted_players['player_position'].apply(lambda x: x.split('_')[0])
df_melted_heroes['hero_num'] = df_melted_heroes['hero_position'].apply(lambda x: x.split('_')[0])

In [ ]:
df_combined = pd.merge(df_melted_players, df_melted_heroes, 
                       left_on=['start_time', 'radiant_win','match_id', 'player_num'], 
                       right_on=['start_time', 'radiant_win','match_id', 'hero_num'])

df_combined = df_combined.sort_values(by='start_time', ascending=False)
df_combined


In [ ]:
# 2. Determine if a player won based on their position and match outcome
df_combined['player_won'] = ((df_combined['player_num'].astype(int) < 5) & df_combined['radiant_win']) | \
                           ((df_combined['player_num'].astype(int) >= 5) & ~df_combined['radiant_win'])


In [ ]:
# 2. Create a key for each account_id and hero combination
# Use hero_name (not hero_num) to identify unique heroes
df_combined['account_hero_key'] = df_combined['account_id'].astype(str) + '_' + df_combined['hero_name'].astype(str)

In [ ]:
# 3. Sort data chronologically
df_sorted = df_combined.sort_values(by=time_col)

In [ ]:
win_rates = {}
for key, group in df_sorted.groupby('account_hero_key'):
    for i, row in group.iterrows():
        match_id = row['match_id']
        player_num = row['player_num']
        current_time = row[time_col]
        
        # Find previous matches for this player-hero combo
        previous_matches = group[group[time_col] < current_time]
        
        # Calculate win rate from previous matches
        if len(previous_matches) > 0:
            previous_10 = previous_matches.sort_values(by=time_col, ascending=False).head(10)
            win_rate = previous_10['player_won'].mean()
        else:
            win_rate = 0.5
            
        win_rates[(match_id, player_num)] = win_rate

In [ ]:
# 5. Apply calculated win rates to dataframe
df_combined['win_rate'] = df_combined.apply(
    lambda row: win_rates.get((row['match_id'], row['player_num']), 0.5),
    axis=1
)

In [ ]:
# 6. Create column names based on player and hero positions
df_combined['player_hero_win_rate_col'] = (
    'player_hero_' + df_combined['player_num'].astype(str) + '_win_rate'
)

In [ ]:
# 7. Create final pivot table with position-based columns
player_hero_features = df_combined.pivot(
    index='match_id', 
    columns='player_hero_win_rate_col', 
    values='win_rate'
).reset_index()

player_hero_features

In [ ]:
player_hero_features

In [ ]:
from database.data_models.features import PlayerHeroFeature

In [ ]:
model_fields = {
    name for name in PlayerHeroFeature.model_fields.keys()
    if not name.startswith('_')
}

model_fields

In [ ]:
def store_player_hero_features(engine, player_hero_feature: pd.DataFrame):
    """
    Store the calculated player-hero win rates to the database
    
    Parameters:
    - engine: SQLAlchemy engine
    - player_hero_feature: DataFrame with match_id and win rate columns
    """
    # Convert DataFrame to list of dictionaries (one dict per match)
    records = player_hero_feature.to_dict(orient="records")
    
    # Create PlayerHeroFeature objects and insert them
    with Session(engine) as session:
        # For each match record
        for record in records:
            # Create a new PlayerHeroFeature instance
            player_hero_feature_obj = PlayerHeroFeature(
                match_id=record["match_id"],
                player_hero_0_win_rate=record["player_hero_0_win_rate"],
                player_hero_1_win_rate=record["player_hero_1_win_rate"],
                player_hero_2_win_rate=record["player_hero_2_win_rate"], 
                player_hero_3_win_rate=record["player_hero_3_win_rate"],
                player_hero_4_win_rate=record["player_hero_4_win_rate"],
                player_hero_128_win_rate=record["player_hero_128_win_rate"],
                player_hero_129_win_rate=record["player_hero_129_win_rate"],
                player_hero_130_win_rate=record["player_hero_130_win_rate"],
                player_hero_131_win_rate=record["player_hero_131_win_rate"],
                player_hero_132_win_rate=record["player_hero_132_win_rate"]
            )
            
            # Use merge instead of add
            session.merge(player_hero_feature_obj)
        
        # Commit all records at once
        try:
            session.commit()
            print(f"Successfully stored {len(records)} player-hero feature records")
        except Exception as e:
            session.rollback()
            print(f"Error storing player-hero features: {str(e)}")
            
store_player_hero_features(engine, player_hero_features)